In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.2 SFT

Supervised fine-tuning trains the model on `(prompt, response)` pairs, but with the
loss **masked on the prompt**: the model is only graded on producing the response.
That masking is the whole trick that turns continuation into instruction-following.

In [ ]:
from torch.nn import functional as F
from pocketlm import CharTokenizer, PocketLM, PocketLMConfig, init_weights

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
prompt, response = "Q: hi A:", " hello"
ids = torch.tensor([tok.encode(prompt + response)])
x = ids[:, :-1]
y = ids[:, 1:].clone()
prompt_len = len(tok.encode(prompt))
y[:, :prompt_len - 1] = -100   # ignore prompt tokens in the loss

torch.manual_seed(0)
model = init_weights(PocketLM(PocketLMConfig(vocab_size=tok.vocab_size, block_size=64,
                                             n_layer=2, n_head=2, n_embd=64)))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
first = None
for _ in range(150 if os.environ.get("POCKETLM_CI") else 300):
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                           ignore_index=-100)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("response loss:", round(first, 3), "->", round(loss.item(), 3))

The loss falls to near zero as the model memorizes the response, on real data you
use many diverse pairs. The mechanism, mask the prompt, train on the response, is
identical at any scale.

In [ ]:
assert loss.item() < first